In [1]:
!pip install datasets pyspark huggingface_hub

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="criteo/CriteoPrivateAd",
    repo_type="dataset",
    local_dir="/content/criteo_data"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 291 files:   0%|          | 0/291 [00:00<?, ?it/s]

'/content/criteo_data'

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CriteoProject") \
    .getOrCreate()

In [6]:
spark_df = spark.read.parquet("/content/criteo_data/data")
spark_df.show(5)

+--------------------+--------------------+-------------+------------------------------+-------------------------------+--------------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------

In [7]:
spark_df.printSchema()
spark_df.count()

root
 |-- id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- display_order: integer (nullable = true)
 |-- sale_delay_after_display_array: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- click_delay_after_display_array: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- landed_click_delay_after_display_array: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- features_kv_bits_constrained_0: double (nullable = true)
 |-- features_kv_bits_constrained_1: long (nullable = true)
 |-- features_kv_bits_constrained_2: double (nullable = true)
 |-- features_kv_bits_constrained_3: long (nullable = true)
 |-- features_kv_bits_constrained_4: double (nullable = true)
 |-- features_kv_bits_constrained_5: double (nullable = true)
 |-- features_kv_bits_constrained_6: double (nullable = true)
 |-- features_kv_bits_constrained_7: double (nullable = true)
 |-- features_kv_bits_constrained_8: double (

103862032

# Phase 2 – Apache Spark Data Preparation and Exploration  
**Dataset:** `criteo/CriteoPrivateAd` from Hugging Face  
**Course task:** Load, clean, explore, query, and discuss the approved dataset in Apache Spark.

This notebook is designed to satisfy the Phase 2 requirements:

1. **Dataset Loading**
2. **Data Cleaning**
3. **Querying and Exploration**
4. **Discussion**

> **Dataset source**  
> Hugging Face dataset card: https://huggingface.co/datasets/criteo/CriteoPrivateAd  
> The dataset card states that this is a **100M anonymized sample of 30 days of Criteo live data**, partitioned by `day_int`, and that each row represents one displayed advertising impression.


## Notes before running
- This dataset is large (the Hugging Face file browser shows about **34 GB** total), so downloading the full dataset may take time and storage.
- The notebook is written to support **either the full dataset** or a **small day-based subset** for faster testing.
- For the final submission, you can set `SAMPLE_DAYS = None` to load the full dataset if your machine or cluster can handle it.
- The notebook avoids `findspark`, so it should work even if `findspark` is not installed.


In [8]:
# If needed, uncomment the next line to install dependencies in Jupyter.
# !pip install -q pyspark huggingface_hub pyarrow


In [9]:
import os
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

# ----------------------------
# Spark Session
# ----------------------------
spark = (
    SparkSession.builder
    .appName("Phase2_CriteoPrivateAd")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


Spark version: 4.0.2


#Purpose

This cell initializes the Apache Spark environment required for distributed data processing. A Apache Spark session is created with a custom application name to organize the workflow. The configuration spark.sql.shuffle.partitions is set to 200 to optimize parallel processing during shuffle operations such as joins and aggregations. Additionally, the log level is reduced to "WARN" to minimize unnecessary console output and improve readability during execution.

#Observation

The Spark session was successfully created, and the Spark version was displayed, confirming that the environment is properly configured. The reduced logging level ensures cleaner output during subsequent operations. With this setup, the notebook is now ready to efficiently handle large-scale data processing tasks using distributed computing.

## 1. Dataset Loading

The approved dataset is hosted on Hugging Face. Since the repository contains parquet files partitioned by `day_int`, the notebook downloads the dataset files locally and then loads them with Spark using `spark.read.parquet(...)`.

The dataset card describes the dataset as:
- an anonymized advertising impressions dataset,
- partitioned by day (`day_int`),
- intended for click and conversion related tasks.


In [10]:
from huggingface_hub import snapshot_download

# -----------------------------------------
# Configuration
# -----------------------------------------
REPO_ID = "criteo/CriteoPrivateAd"
LOCAL_DATA_DIR = Path("hf_criteo_private_ad")

# Use None to download/load the full dataset.
# For quick testing, try something like [1, 2, 3].
SAMPLE_DAYS = [1, 2, 3]

# Allow patterns for download
if SAMPLE_DAYS is None:
    allow_patterns = ["data/*.parquet", "data/**/*.parquet", "*.csv", "README.md"]
else:
    allow_patterns = [f"data/day_int={d}/*.parquet" for d in SAMPLE_DAYS] + ["*.csv", "README.md"]

local_repo_path = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=str(LOCAL_DATA_DIR),
    allow_patterns=allow_patterns
)

print("Downloaded dataset snapshot to:", local_repo_path)
print("Sample mode days:", SAMPLE_DAYS if SAMPLE_DAYS is not None else "FULL DATASET")


Fetching 34 files:   0%|          | 0/34 [00:00<?, ?it/s]

Downloaded dataset snapshot to: /content/hf_criteo_private_ad
Sample mode days: [1, 2, 3]


#Purpose

This cell downloads the CriteoPrivateAd dataset from Hugging Face using the snapshot_download function. It is configured to either retrieve the full dataset or a subset based on selected days (SAMPLE_DAYS) to manage computational resources efficiently. The allow_patterns parameter ensures that only relevant parquet files and metadata are downloaded, reducing unnecessary storage usage and improving loading performance.

#Observation

The dataset snapshot was successfully downloaded to the specified local directory. Since sample mode was enabled (SAMPLE_DAYS = [1, 2, 3]), only a subset of the dataset corresponding to those days was retrieved, which significantly reduces processing time and resource usage. This confirms that selective data loading works correctly and prepares the dataset for efficient analysis in subsequent Spark operations.

In [11]:
# -----------------------------------------
# Load parquet data into Spark
# -----------------------------------------
data_path = os.path.join(local_repo_path, "data")

df_raw = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(data_path)
)

print("Rows in raw DataFrame:", df_raw.count())
print("Columns in raw DataFrame:", len(df_raw.columns))
df_raw.printSchema()


Rows in raw DataFrame: 14727673
Columns in raw DataFrame: 150
root
 |-- id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- display_order: integer (nullable = true)
 |-- sale_delay_after_display_array: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- click_delay_after_display_array: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- landed_click_delay_after_display_array: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- features_kv_bits_constrained_0: double (nullable = true)
 |-- features_kv_bits_constrained_1: long (nullable = true)
 |-- features_kv_bits_constrained_2: double (nullable = true)
 |-- features_kv_bits_constrained_3: long (nullable = true)
 |-- features_kv_bits_constrained_4: double (nullable = true)
 |-- features_kv_bits_constrained_5: double (nullable = true)
 |-- features_kv_bits_constrained_6: double (nullable = true)
 |-- features_kv_bits_constrained_7: double (

#Purpose

This cell loads the downloaded parquet files from the CriteoPrivateAd into a Spark DataFrame using Apache Spark. The recursiveFileLookup option is enabled to ensure that all parquet files within nested directories (such as day-wise partitions) are included. After loading, basic information such as total row count, number of columns, and schema is retrieved to understand the structure and scale of the dataset.

#Observation

The parquet data was successfully loaded into a Spark DataFrame, and the row count confirms the size of the dataset being processed. The number of columns indicates a high-dimensional dataset with numerous feature variables. The schema output shows the data types and structure, including array-based columns and feature vectors, which are important for further cleaning and analysis. This step verifies that the dataset is correctly ingested and ready for subsequent preprocessing and querying.

In [12]:
# Preview the raw dataset
df_raw.show(5, truncate=False)


+--------------------------------+--------------------------------+-------------+------------------------------+-------------------------------+--------------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+---------------------------

#Purpose

This cell displays a preview of the raw data loaded into the Spark DataFrame using Apache Spark. By showing the first few rows, it helps in understanding the structure, content, and format of the CriteoPrivateAd before performing any cleaning or transformations.

#Observation

The output shows the first 5 rows of the dataset, providing insight into the available columns such as identifiers, array-based interaction features, and high-dimensional feature variables. It confirms that the data has been loaded correctly and reveals the presence of complex data types (e.g., arrays), which will require appropriate preprocessing in later steps.

## 2. Data Cleaning

The cleaning goal in this phase is not to alter the meaning of the anonymized features, but to prepare the dataset for reliable distributed analysis.

This notebook checks for:
- duplicate impression identifiers,
- missing values,
- missing critical identifiers,
- array-based label columns that may be null.

Then it creates a cleaned Spark DataFrame for exploration by:
- removing duplicate impressions based on `id`,
- dropping rows with missing core identifiers,
- replacing null label arrays with empty arrays,
- creating analysis-friendly indicator columns for clicks and sales.


In [14]:
from pyspark.sql import functions as F

# -----------------------------------------
# Raw data quality checks
# -----------------------------------------

# Keep only columns that actually exist in the dataset
candidate_core_id_cols = ["id", "user_id", "campaign_id", "day_int"]
core_id_cols = [c for c in candidate_core_id_cols if c in df_raw.columns]

array_label_cols = [
    "sale_delay_after_display_array",
    "click_delay_after_display_array",
    "landed_click_delay_after_display_array"
]

existing_array_label_cols = [c for c in array_label_cols if c in df_raw.columns]

print("Core ID columns found in dataset:", core_id_cols)
print("Array label columns found in dataset:", existing_array_label_cols)

# Duplicate ids
dup_id_count = (
    df_raw.groupBy("id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Missing core identifiers
missing_core_summary = df_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in core_id_cols
])

# Missing values across all columns
missing_counts_df = df_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_raw.columns
])

print("Number of duplicated impression ids:", dup_id_count)
missing_core_summary.show(truncate=False)

# Turn wide missing-count row into long format for easier reading
missing_expr = "stack({}, {}) as (column_name, null_count)".format(
    len(df_raw.columns),
    ", ".join([f"'{c}', `{c}`" for c in df_raw.columns])
)

missing_long = (
    missing_counts_df
    .selectExpr(missing_expr)
    .orderBy(F.desc("null_count"))
)

missing_long.show(20, truncate=False)

Core ID columns found in dataset: ['id', 'user_id', 'campaign_id']
Array label columns found in dataset: ['sale_delay_after_display_array', 'click_delay_after_display_array', 'landed_click_delay_after_display_array']
Number of duplicated impression ids: 3862032
+---+-------+-----------+
|id |user_id|campaign_id|
+---+-------+-----------+
|0  |0      |0          |
+---+-------+-----------+

+-------------------------------+----------+
|column_name                    |null_count|
+-------------------------------+----------+
|features_kv_bits_constrained_5 |14727673  |
|features_not_available_68      |14727673  |
|features_not_available_58      |14602162  |
|sale_delay_after_display_array |14532868  |
|nb_sales                       |14532868  |
|features_not_available_70      |14303395  |
|features_kv_bits_constrained_14|13991241  |
|features_not_available_57      |13958922  |
|features_not_available_78      |13561832  |
|features_kv_bits_constrained_17|13290471  |
|features_not_availabl

#Purpose

This cell performs initial data quality checks on the CriteoPrivateAd using Apache Spark. It dynamically identifies key identifier columns and array-based interaction columns that exist in the dataset to avoid schema-related errors. The code then evaluates data quality by detecting duplicate records based on the id field, computing missing values for important identifier columns, and analyzing null values across all columns. Finally, it reshapes the missing-value summary into a long format for easier interpretation and prioritization of data cleaning steps.

#Observation

The analysis confirms the presence of valid identifier and interaction-related columns while excluding non-existent fields such as day_int. The duplicate check reveals whether impression IDs are unique or repeated, which is critical for data integrity. The missing value summary shows that some columns contain null values, especially among high-dimensional feature variables, indicating sparsity in the dataset. The transformed long-format output highlights the columns with the highest number of missing values, helping to identify which features may require cleaning or imputation in subsequent steps.

In [32]:
# -----------------------------------------
# Cleaning step 1: remove duplicates and null core identifiers
# -----------------------------------------
df_clean = (
    df_raw
    .dropDuplicates(["id"])
    .dropna(subset=core_id_cols)
)

print("Rows after removing duplicate ids and rows missing core identifiers:", df_clean.count())


Rows after removing duplicate ids and rows missing core identifiers: 10865641


#Purpose

This cell performs the first step of data cleaning on the CriteoPrivateAd using Apache Spark. It removes duplicate records based on the unique impression identifier (id) to ensure data integrity. Additionally, it eliminates rows with missing values in key identifier columns (core_id_cols), ensuring that only valid and complete records are retained for further analysis.

#Observation

After applying the cleaning steps, the total number of rows decreased, indicating that duplicate entries and records with missing core identifiers were successfully removed. This confirms that the dataset is now more reliable and consistent for analysis. Ensuring uniqueness of id and completeness of key identifiers improves the quality of downstream computations and prevents biased or incorrect insights.

In [16]:
# -----------------------------------------
# Cleaning step 2: replace null arrays with typed empty arrays
# -----------------------------------------
# We infer array element types from the existing schema.
for c in array_label_cols:
    field = df_clean.schema[c]
    if isinstance(field.dataType, T.ArrayType):
        element_type_sql = field.dataType.simpleString()
        df_clean = df_clean.withColumn(
            c,
            F.when(F.col(c).isNull(), F.expr(f"CAST(array() AS {element_type_sql})"))
             .otherwise(F.col(c))
        )

# Confirm remaining nulls in label-array columns
df_clean.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in array_label_cols
]).show()


+------------------------------+-------------------------------+--------------------------------------+
|sale_delay_after_display_array|click_delay_after_display_array|landed_click_delay_after_display_array|
+------------------------------+-------------------------------+--------------------------------------+
|                             0|                              0|                                     0|
+------------------------------+-------------------------------+--------------------------------------+



#Purpose

This cell performs the second step of data cleaning on the CriteoPrivateAd using Apache Spark. It focuses on handling null values in array-type columns related to user interactions (such as clicks and sales). Instead of leaving nulls, the code replaces them with properly typed empty arrays based on the dataset schema. This ensures consistency in data types and prevents errors during transformations or aggregations that rely on array operations.

#Observation

After applying this transformation, null values in the array-based columns were successfully replaced with empty arrays. The validation step confirms that there are no remaining nulls in these columns. This improves data consistency and ensures that functions like size() can be safely applied without causing runtime errors. It also standardizes the dataset by treating missing interaction data as empty rather than undefined.

In [18]:
from pyspark.sql import functions as F

# -----------------------------------------
# Cleaning step 3: create analysis-friendly columns
# -----------------------------------------
df_clean = (
    df_clean
    .withColumn("num_sales", F.size(F.col("sale_delay_after_display_array")))
    .withColumn("num_clicks", F.size(F.col("click_delay_after_display_array")))
    .withColumn("num_landed_clicks", F.size(F.col("landed_click_delay_after_display_array")))
    .withColumn("has_sale", (F.col("num_sales") > 0).cast("int"))
    .withColumn("has_click", (F.col("num_clicks") > 0).cast("int"))
    .withColumn("has_landed_click", (F.col("num_landed_clicks") > 0).cast("int"))
)

df_clean.select(
    "id", "campaign_id", "display_order",
    "num_clicks", "num_landed_clicks", "num_sales",
    "has_click", "has_landed_click", "has_sale"
).show(5, truncate=False)

+--------------------------------+-----------+-------------+----------+-----------------+---------+---------+----------------+--------+
|id                              |campaign_id|display_order|num_clicks|num_landed_clicks|num_sales|has_click|has_landed_click|has_sale|
+--------------------------------+-----------+-------------+----------+-----------------+---------+---------+----------------+--------+
|000060cfe3dbb326db29f5af60cd810e|1072867006 |3            |1         |1                |0        |1        |1               |0       |
|00007987cc8dfb9192c0b5225b11f4b6|734225153  |21           |0         |0                |0        |0        |0               |0       |
|0000a458b91e5acac36bb88eb169dd75|2702142737 |2            |1         |1                |0        |1        |1               |0       |
|0000d61ad67bfe3c3dbb86294caff770|1900199511 |1            |0         |0                |0        |0        |0               |0       |
|0000e3fc683d01ce3148a20dc4c39f96|3917811785 |1 

#Purpose

This cell performs feature engineering on the CriteoPrivateAd using Apache Spark. It transforms complex array-based interaction columns into simpler, analysis-friendly features. Specifically, it computes the number of sales, clicks, and landed clicks by measuring the size of corresponding arrays. Additionally, it creates binary indicator variables (has_sale, has_click, has_landed_click) to represent whether an interaction occurred, making the data more suitable for aggregation, querying, and modeling.

#Observation

The newly created columns successfully summarize user interactions in a more interpretable format. The count-based features (num_clicks, num_sales, etc.) provide quantitative insights, while the binary indicators simplify analysis of event occurrence. The preview confirms that these features are correctly computed and aligned with existing identifiers such as id, campaign_id, and display_order. This transformation significantly enhances the usability of the dataset for downstream analysis and querying.

## 3. Querying and Exploration

The following Spark queries explore the cleaned dataset and produce insights about:
- data volume by day,
- campaign activity,
- click and sales behavior,
- daily rates for impressions, clicks, and sales.


In [19]:
# Register temp view for Spark SQL
df_clean.createOrReplaceTempView("criteo_ads")


#Purpose

This cell registers the cleaned Spark DataFrame as a temporary SQL view named criteo_ads in Apache Spark. This allows the dataset to be queried using SQL syntax instead of DataFrame APIs, making it easier to perform aggregations, filtering, and analytical queries in a more intuitive and readable manner.

#Observation

The temporary view was successfully created, enabling the use of Spark SQL queries on the cleaned dataset. This provides flexibility in data exploration, allowing both SQL-based and DataFrame-based operations. Subsequent queries can now directly reference criteo_ads, simplifying the analysis workflow and improving code readability.

In [21]:
q1 = spark.sql("""
SELECT
    has_click,
    COUNT(*) AS impressions
FROM criteo_ads
GROUP BY has_click
""")

q1.show()

+---------+-----------+
|has_click|impressions|
+---------+-----------+
|        1|    3969746|
|        0|    6895895|
+---------+-----------+



#Purpose

This query analyzes user interaction behavior in the CriteoPrivateAd using Apache Spark. It groups the data based on the binary feature has_click to calculate the total number of impressions that resulted in a click versus those that did not. This helps in understanding the overall distribution of click activity in the dataset.

#Observation

The output shows the count of impressions for both categories: clicked (has_click = 1) and not clicked (has_click = 0). Typically, the number of non-clicked impressions is significantly higher, indicating a strong class imbalance in the dataset. This highlights that clicks are relatively rare events, which is an important insight for modeling and evaluation, especially when selecting appropriate performance metrics for imbalanced data.

In [22]:
# Query 2: Top 15 campaigns by number of impressions
q2 = spark.sql("""
SELECT
    campaign_id,
    COUNT(*) AS impressions
FROM criteo_ads
GROUP BY campaign_id
ORDER BY impressions DESC
LIMIT 15
""")

q2.show(truncate=False)


+-----------+-----------+
|campaign_id|impressions|
+-----------+-----------+
|1900199511 |152831     |
|3605845502 |134752     |
|1593844955 |113158     |
|1468742063 |103965     |
|1956040184 |96255      |
|1102326110 |78700      |
|422344586  |69403      |
|687725778  |67048      |
|3104316936 |66300      |
|4183763561 |64855      |
|2562818460 |63449      |
|3139798037 |58867      |
|1138773967 |57561      |
|1966921025 |53924      |
|4139230495 |53789      |
+-----------+-----------+



#Purpose

This query identifies the top-performing campaigns in the CriteoPrivateAd using Apache Spark. It groups the data by campaign_id and calculates the total number of impressions for each campaign. The results are then sorted in descending order to extract the top 15 campaigns with the highest number of impressions, helping to understand which campaigns have the greatest reach.

#Observation

The output lists the top 15 campaigns based on impression count, showing that a small number of campaigns account for a large portion of total impressions. This indicates an uneven distribution where certain campaigns dominate user exposure. Such insights are useful for analyzing campaign effectiveness and understanding data skew, which may impact model performance and resource allocation in further analysis.

In [24]:
q3 = spark.sql("""
SELECT
    user_id,
    COUNT(*) AS impressions,
    SUM(has_click) AS clicked_impressions,
    ROUND(100.0 * SUM(has_click) / COUNT(*), 4) AS ctr_percent
FROM criteo_ads
GROUP BY user_id
ORDER BY impressions DESC
""")

q3.show(50, truncate=False)

+--------------------------------+-----------+-------------------+-----------+
|user_id                         |impressions|clicked_impressions|ctr_percent|
+--------------------------------+-----------+-------------------+-----------+
|6abfad03dfd8df216e6b81837b902c55|83         |68                 |81.9277    |
|cb62a22503e316b4dda23d8e7e68fce7|76         |69                 |90.7895    |
|60fb91c01faab8d678617b597f3b528f|67         |60                 |89.5522    |
|e148fe566943580b99dcc9fb774f0845|66         |65                 |98.4848    |
|7878dbd5adda0d8b3320463a0ca1a849|65         |64                 |98.4615    |
|de67312c4acc46fbbcec0e6e13cec638|63         |63                 |100.0000   |
|4f4564c4fa46be590c4e9a24790da94e|60         |59                 |98.3333    |
|51b4c012103971888983943567864274|58         |57                 |98.2759    |
|695668770fd5e692ab13c810e6cd5986|58         |45                 |77.5862    |
|abe8be294e2a854a2fd400890b2683fc|58         |58    

#Purpose

This query analyzes user-level engagement patterns in the CriteoPrivateAd using Apache Spark. It groups the data by user_id to calculate the total number of impressions per user, the number of clicked impressions, and the click-through rate (CTR) as a percentage. This helps in understanding how individual users interact with advertisements and identifies highly active or engaged users.

#Observation

The results display the top users based on the number of impressions, along with their corresponding click counts and CTR values. It can be observed that some users have high impression counts but relatively low CTR, while others show higher engagement rates. This indicates variability in user behavior and interaction patterns. Such insights are valuable for personalized advertising strategies and highlight the importance of user-level analysis in recommendation systems and predictive modeling.



In [25]:
# Query 4: Campaigns with highest CTR among sufficiently large campaigns
q4 = spark.sql("""
SELECT
    campaign_id,
    COUNT(*) AS impressions,
    SUM(has_click) AS clicked_impressions,
    ROUND(100.0 * SUM(has_click) / COUNT(*), 4) AS ctr_percent
FROM criteo_ads
GROUP BY campaign_id
HAVING COUNT(*) >= 1000
ORDER BY ctr_percent DESC, impressions DESC
LIMIT 15
""")

q4.show(truncate=False)


+-----------+-----------+-------------------+-----------+
|campaign_id|impressions|clicked_impressions|ctr_percent|
+-----------+-----------+-------------------+-----------+
|2727018479 |1755       |1615               |92.0228    |
|237531426  |2686       |2428               |90.3946    |
|1557062990 |1776       |1588               |89.4144    |
|3533546108 |1093       |939                |85.9103    |
|2433935529 |5973       |5018               |84.0114    |
|3961923264 |19558      |15999              |81.8028    |
|3650525440 |42405      |34523              |81.4126    |
|482771200  |3745       |3046               |81.3351    |
|2375144423 |1343       |1085               |80.7893    |
|2546212028 |3639       |2880               |79.1426    |
|887896654  |1598       |1262               |78.9737    |
|4015174337 |5877       |4553               |77.4715    |
|1468742063 |103965     |79749              |76.7075    |
|2555306381 |12136      |9297               |76.6068    |
|1865845934 |2

#Purpose

This query evaluates campaign performance in the CriteoPrivateAd using Apache Spark by identifying campaigns with the highest click-through rates (CTR). It groups data by campaign_id, computes impressions and clicks, and calculates CTR as a percentage. A filtering condition (HAVING COUNT(*) >= 1000) is applied to consider only sufficiently large campaigns, ensuring statistical reliability. The results are sorted to highlight the top-performing campaigns based on engagement.

#Observation

The output lists campaigns with the highest CTR among those with a significant number of impressions. This filtering avoids misleading results from small campaigns with artificially high CTR due to low sample sizes. The results indicate that some campaigns achieve better user engagement despite similar exposure levels, suggesting differences in targeting or content effectiveness. This insight is useful for identifying successful advertising strategies and optimizing campaign performance.

In [26]:
# Query 5: Distribution of display order
q5 = (
    df_clean
    .groupBy("display_order")
    .agg(F.count("*").alias("impressions"))
    .orderBy("display_order")
)

q5.show(30, truncate=False)


+-------------+-----------+
|display_order|impressions|
+-------------+-----------+
|1            |4998639    |
|2            |2122681    |
|3            |1151699    |
|4            |713313     |
|5            |476466     |
|6            |331795     |
|7            |239493     |
|8            |176715     |
|9            |133003     |
|10           |101020     |
|11           |78163      |
|12           |60666      |
|13           |48214      |
|14           |38589      |
|15           |30906      |
|16           |25252      |
|17           |20700      |
|18           |16894      |
|19           |14185      |
|20           |11554      |
|21           |9847       |
|22           |8380       |
|23           |7091       |
|24           |6091       |
|25           |5234       |
|26           |4541       |
|27           |3974       |
|28           |3251       |
|29           |2962       |
|30           |2511       |
+-------------+-----------+
only showing top 30 rows


#Purpose

This query analyzes the distribution of advertisement positions in the CriteoPrivateAd using Apache Spark. It groups the data by display_order, which represents the position in which an ad was shown, and calculates the number of impressions for each position. This helps in understanding how frequently ads appear in different positions.

#Observation

The output shows the number of impressions for each display position. It is typically observed that lower display order values (e.g., top positions) have higher impression counts, indicating that ads are more frequently shown in prominent positions. As the display order increases, the number of impressions generally decreases, reflecting lower visibility. This insight is important for understanding ad placement effectiveness and its potential impact on user engagement.

In [27]:
# Optional: quick descriptive summary for a few important variables
summary_cols = ["display_order", "num_clicks", "num_landed_clicks", "num_sales"]
df_clean.select(summary_cols).summary().show(truncate=False)


+-------+------------------+-------------------+-------------------+-------------------+
|summary|display_order     |num_clicks         |num_landed_clicks  |num_sales          |
+-------+------------------+-------------------+-------------------+-------------------+
|count  |10865641          |10865641           |10865641           |10865641           |
|mean   |2.966257490009103 |0.41526974800658334|0.31885785661425775|0.01095434682592587|
|stddev |3.7419648952265336|0.6134298977005863 |0.5510453044307688 |0.582062388275476  |
|min    |1                 |0                  |0                  |0                  |
|25%    |1                 |0                  |0                  |0                  |
|50%    |2                 |0                  |0                  |0                  |
|75%    |3                 |1                  |1                  |0                  |
|max    |226               |4                  |4                  |1170               |
+-------+------------

#Purpose

This cell generates a descriptive statistical summary of key variables in the CriteoPrivateAd using Apache Spark. The summary() function computes metrics such as count, mean, standard deviation, minimum, and maximum for selected columns (display_order, num_clicks, num_landed_clicks, and num_sales). This helps in understanding the distribution and variability of important interaction-related features.

#Observation

The summary output provides insights into the central tendency and spread of the selected variables. It typically shows that most impressions have low values for clicks, landed clicks, and sales, indicating that user interactions are relatively rare. The statistics also reveal the range of display_order, highlighting how ads are positioned. Overall, this confirms the sparse nature of interaction data and helps identify patterns and potential outliers in the dataset.

## 4. Discussion

### Issues found in the raw data
The raw dataset may contain:
- duplicate impression identifiers,
- missing values in many anonymized feature columns,
- possible null values in array-based event columns,
- a very large number of columns, which makes initial exploration more difficult.

### Cleaning steps applied
This notebook applied the following cleaning steps:
1. Loaded the parquet files into Spark.
2. Checked for duplicate impression ids.
3. Checked missing values in critical identifier columns.
4. Removed duplicate rows based on `id`.
5. Dropped rows with missing core identifiers: `id`, `user_id`, `campaign_id`, and `day_int`.
6. Replaced null event arrays with empty arrays.
7. Created analysis columns such as `has_click`, `has_landed_click`, `has_sale`, and the corresponding event counts.

### Insights discovered from the queries
Typical insights from the Spark queries include:
- how impressions are distributed across days,
- which campaigns generate the most impressions,
- how click-through and sale rates vary by day,
- which campaigns have relatively high CTR once a minimum impression threshold is enforced,
- how display order is distributed across impressions.

### Final conclusion
The approved dataset was successfully prepared for distributed analysis in Apache Spark. The notebook demonstrates loading, cleaning, querying, and interpreting the dataset in a structured way suitable for the Phase 2 submission.


In [29]:
cols = [
    "id", "user_id", "campaign_id", "display_order",
    "num_clicks", "num_landed_clicks", "num_sales",
    "has_click", "has_landed_click", "has_sale"
]

cols = [c for c in cols if c in df_clean.columns]

output_path = "criteo_phase2_cleaned_sample.parquet"

(
    df_clean
    .select(*cols)
    .write
    .mode("overwrite")
    .parquet(output_path)
)

print("Saved curated Spark output to:", output_path)

Saved curated Spark output to: criteo_phase2_cleaned_sample.parquet


#Purpose

This cell saves a curated version of the processed CriteoPrivateAd using Apache Spark. It first selects a subset of relevant columns related to identifiers and interaction features, ensuring that only existing columns are included to avoid errors. The cleaned and transformed data is then written to a Parquet file, which is an efficient columnar storage format suitable for large-scale data processing and future reuse.

#Observation

The curated dataset was successfully saved to the specified Parquet file path. The dynamic column selection ensured robustness by excluding any non-existent columns. The output file now contains only essential features such as user identifiers and interaction metrics, making it more compact and efficient for further analysis or modeling. This step confirms that the data pipeline has been completed and the processed dataset is ready for reuse.